In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile
from tqdm import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')
from math import ceil
from collections import defaultdict


import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input          #importing dependencies
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, concatenate, Bidirectional, Dot, Activation, RepeatVector, Lambda
from tensorflow.keras.utils import to_categorical, plot_model

from nltk.translate.bleu_score import corpus_bleu
import json
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

# Define the zip file path
zip_path = "/content/drive/MyDrive/M.L./Copy of hindi-visual-genome-11.zip"
extract_to = "/content/hindi_visual_genome"

# Create the destination folder if it doesn't exist
os.makedirs(extract_to, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("✅ Zip file extracted to:", extract_to)


✅ Zip file extracted to: /content/hindi_visual_genome


In [ ]:
import json

input_file = "/content/hindi_visual_genome/hindi-visual-genome-11/hindi-visual-genome-11/hindi-visual-genome-train.txt"
output_file = "/content/image_ids_and_captions.json"

converted_data = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split('\t')

        if len(parts) < 7:
            # Skip lines with incomplete data
            continue

        image_id = parts[0].strip()
        english_caption = parts[-2].strip()
        hindi_caption = parts[-1].strip()

        converted_data.append({
            "image_id": image_id,
            "english_caption": english_caption,
            "hindi_caption": hindi_caption
        })

# Save to JSON
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=2)

print(f"✅ Converted JSON saved to {output_file}")


✅ Converted JSON saved to /content/image_ids_and_captions.json


In [ ]:
import json
import re

# Load JSON file
json_path = "/content/image_ids_and_captions.json"
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Hindi text cleaner function
def clean_hindi_text(text):
    # Remove unwanted punctuation (except ।)
    text = re.sub(r"[\"!#$%&()*+,-./:;<=>?@[\\]^_`{|}~]", "", text)
    # Normalize white spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Extract and clean all hindi captions
cleaned_lines = []
for entry in data:
    hindi_caption = entry.get("hindi_caption", "").strip()
    if hindi_caption:
        cleaned_caption = clean_hindi_text(hindi_caption)
        cleaned_lines.append(cleaned_caption)

# Save to text file
output_path = "/content/hindi_captions_cleaned.txt"
with open(output_path, "w", encoding="utf-8") as f:
    for line in cleaned_lines:
        f.write(line + "\n")

print("✅ Cleaned Hindi captions saved to", output_path)


✅ Cleaned Hindi captions saved to /content/hindi_captions_cleaned.txt


In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input="/content/hindi_captions_cleaned.txt",
    model_prefix="/content/hindi_unigram_5000",
    vocab_size=5000,
    model_type="unigram",
    character_coverage=1.0,
    train_extremely_large_corpus=True  # helps with small datasets
)


In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("/content/hindi_unigram_5000.model")

sample = "एक सुंदर पक्षी उड़ रहा है।"
print("📤 Tokens:", sp.encode(sample, out_type=str))
print("📥 IDs:", sp.encode(sample, out_type=int))

📤 Tokens: ['▁एक', '▁सुंदर', '▁पक्षी', '▁उड़', '▁रहा', '▁है', '।']
📥 IDs: [3, 524, 102, 464, 25, 5, 11]


In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input="/content/hindi_captions_cleaned.txt",
    model_prefix="/content/hindi_unigram_3000",
    vocab_size=3000,
    model_type="unigram",
    character_coverage=1.0,
    train_extremely_large_corpus=True  # helps with small datasets
)


In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("/content/hindi_unigram_5000.model")

sample = "मैंने कल बाजार से कुछ फल और सब्जियाँ खरीदीं।"
print("📤 Tokens:", sp.encode(sample, out_type=str))
print("📥 IDs:", sp.encode(sample, out_type=int))

📤 Tokens: ['▁मै', 'ं', 'ने', '▁कल', '▁बा', 'जार', '▁से', '▁कुछ', '▁फल', '▁और', '▁सब्जियाँ', '▁खरीद', 'ीं', '।']
📥 IDs: [1006, 598, 75, 4267, 237, 1593, 28, 270, 282, 15, 2478, 3177, 2251, 11]
